In [1]:
# Initialize Environment

!conda activate vesselfm

In [ ]:
#!/usr/bin/env python

import os
import argparse
from pathlib import Path

import numpy as np
import nibabel as nib
from scipy.ndimage import zoom


def resample_to_spacing(data, src_spacing, tgt_spacing, order):
    """
    Data: np.ndarray with shape (X, Y, Z) or (X, Y, Z, C)
    src_spacing, tgt_spacing: iterable of length 3
    order: interpolation order (3 = cubic for image, 0 = nearest for label)
    """
    src_spacing = np.array(src_spacing, dtype=np.float32)
    tgt_spacing = np.array(tgt_spacing, dtype=np.float32)
    zoom_factors = src_spacing / tgt_spacing

    if data.ndim == 3:
        factors = zoom_factors
    elif data.ndim == 4:
        factors = (*zoom_factors, 1.0)  # don't scale channels
    else:
        raise ValueError(f"Unsupported data ndim {data.ndim}, expected 3 or 4.")

    resampled = zoom(data, factors, order=order)
    return resampled


def compute_label_bbox(label, margin=0):
    """
    Compute bounding box of non-zero labels, with margin in voxels.
    label: np.ndarray (X, Y, Z) integer labels
    returns: slices or None if no foreground
    """
    if np.max(label) == 0:
        return None

    coords = np.where(label > 0)
    xmin, xmax = coords[0].min(), coords[0].max()
    ymin, ymax = coords[1].min(), coords[1].max()
    zmin, zmax = coords[2].min(), coords[2].max()

    xmin = max(xmin - margin, 0)
    ymin = max(ymin - margin, 0)
    zmin = max(zmin - margin, 0)

    xmax = min(xmax + margin, label.shape[0] - 1)
    ymax = min(ymax + margin, label.shape[1] - 1)
    zmax = min(zmax + margin, label.shape[2] - 1)

    return (slice(xmin, xmax + 1),
            slice(ymin, ymax + 1),
            slice(zmin, zmax + 1))


def zscore_normalize(img, mask=None, eps=1e-8):
    """
    Z-score normalization with mask.
    img: np.ndarray float
    mask: boolean array or None
    """
    if mask is None:
        mask = np.ones_like(img, dtype=bool)
    vals = img[mask]
    if vals.size == 0:
        return img
    mean = vals.mean()
    std = vals.std()
    if std < eps:
        std = eps
    img = (img - mean) / std
    return img


def percentile_normalize(img, p_lo=0.5, p_hi=99.5, mask=None, eps=1e-8):
    """
    Map intensities between [p_lo, p_hi] percentiles to [0,1].
    Everything below p_lo goes to 0, above p_hi to 1.
    Usually mask = (img != 0) to ignore the air background.
    """
    if mask is None:
        mask = np.ones_like(img, dtype=bool)

    vals = img[mask]
    if vals.size == 0:
        return img

    lo = np.percentile(vals, p_lo)
    hi = np.percentile(vals, p_hi)

    if hi - lo < eps:
        # almost constant volume, nothing sensible to do
        return img

    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + eps)
    return img


def preprocess_pair(
    img_path,
    lbl_path,
    out_img_dir,
    out_lbl_dir,
    target_spacing,
    clip_range=None,
    do_zscore=False,
    crop_mode="label",
    crop_margin=10,
    percentile_norm=False,
    percentile_range=(0.5, 99.5),
):
    """
    Preprocess one image+label pair and save as .nii.gz.

    img_path, lbl_path: Path objects (input .nii/.nii.gz)
    out_img_dir, out_lbl_dir: Path objects (output directories)
    """
    print(f"Processing: {img_path.name}")

    img_nii = nib.load(str(img_path))
    img = img_nii.get_fdata().astype(np.float32)
    src_spacing = img_nii.header.get_zooms()[:3]

    # Labels
    if lbl_path is not None:
        lbl_nii = nib.load(str(lbl_path))
        lbl = lbl_nii.get_fdata().astype(np.int16)
    else:
        lbl_nii = None
        lbl = None

    # Resample image and label to target spacing
    if target_spacing is not None:
        img = resample_to_spacing(img, src_spacing, target_spacing, order=3)
        if lbl is not None:
            lbl = resample_to_spacing(lbl, src_spacing, target_spacing, order=0)
        spacing = target_spacing
    else:
        spacing = src_spacing

    # Intensity normalisation
    # HU clipping
    if clip_range is not None:
        lo, hi = clip_range
    else:
        lo, hi = -1000.0, 600.0  # default chest CT window

    img = np.clip(img, lo, hi)

    # Choose one of: percentile mapping, z-score, or simple [lo,hi] to [0,1]
    if percentile_norm:
        # Ignore pure-air voxels when computing percentiles
        mask = img != lo
        p_lo, p_hi = percentile_range
        img = percentile_normalize(img, p_lo=p_lo, p_hi=p_hi, mask=mask)
    elif do_zscore:
        mask = img != lo
        img = zscore_normalize(img, mask=mask)
    else:
        # Simple linear windowing [lo,hi] to [0,1]
        img = (img - lo) / (hi - lo + 1e-8)

    img = img.astype(np.float32)

    # Cropping

    # For inference, use crop_mode="none" so volume size stays global.
    if crop_mode == "label" and lbl is not None:
        bbox = compute_label_bbox(lbl, margin=crop_margin)
        if bbox is not None:
            img = img[bbox]
            lbl = lbl[bbox]
    elif crop_mode == "none":
        # No cropping
        pass
    elif crop_mode == "body":
        # Simple body mask
        mask = img != 0
        if np.any(mask):
            coords = np.where(mask)
            xmin, xmax = coords[0].min(), coords[0].max()
            ymin, ymax = coords[1].min(), coords[1].max()
            zmin, zmax = coords[2].min(), coords[2].max()
            xmin = max(xmin - crop_margin, 0)
            ymin = max(ymin - crop_margin, 0)
            zmin = max(zmin - crop_margin, 0)
            xmax = min(xmax + crop_margin, img.shape[0] - 1)
            ymax = min(ymax + crop_margin, img.shape[1] - 1)
            zmax = min(zmax + crop_margin, img.shape[2] - 1)
            bbox = (slice(xmin, xmax + 1),
                    slice(ymin, ymax + 1),
                    slice(zmin, zmax + 1))
            img = img[bbox]
            if lbl is not None:
                lbl = lbl[bbox]
    else:
        raise ValueError(f"Unknown crop_mode: {crop_mode}")

    affine = np.eye(4, dtype=np.float32)
    affine[0, 0] = spacing[0]
    affine[1, 1] = spacing[1]
    affine[2, 2] = spacing[2]

    out_img_dir.mkdir(parents=True, exist_ok=True)
    if out_lbl_dir is not None:
        out_lbl_dir.mkdir(parents=True, exist_ok=True)

    out_img_path = out_img_dir / img_path.name
    img_out = nib.Nifti1Image(img.astype(np.float32), affine)
    nib.save(img_out, str(out_img_path))

    if lbl is not None and out_lbl_dir is not None:
        out_lbl_path = out_lbl_dir / lbl_path.name
        lbl_out = nib.Nifti1Image(lbl.astype(np.int16), affine)
        nib.save(lbl_out, str(out_lbl_path))

    print(f"  -> saved image to {out_img_path}")
    if lbl is not None and out_lbl_dir is not None:
        print(f"  -> saved label to {out_lbl_path}")


def main():
    parser = argparse.ArgumentParser(
        description="Preprocessing for VesselFM Adaptation"
    )
    parser.add_argument("--images", type=str, required=True,
                        help="Directory with input .nii.gz images")
    parser.add_argument("--labels", type=str, default=None,
                        help="Directory with input .nii.gz labels (optional)")
    parser.add_argument("--out_images", type=str, required=True,
                        help="Output directory for preprocessed images")
    parser.add_argument("--out_labels", type=str, default=None,
                        help="Output directory for preprocessed labels (optional)")
    parser.add_argument(
        "--target_spacing",
        type=float,
        nargs=3,
        default=[1.0, 1.0, 1.0],
        help="Target voxel spacing (sx sy sz). Use 1.0 1.0 1.0 or dataset median."
    )
    parser.add_argument(
        "--percentile_norm",
        action="store_true",
        help="Use per-volume percentile mapping after HU clipping instead of plain [lo,hi]->[0,1] or z-score."
    )
    parser.add_argument(
        "--percentile_range",
        type=float,
        nargs=2,
        default=[0.5, 99.5],
        help="Lower/upper percentiles for --percentile_norm (default 0.5 99.5)."
    )
    parser.add_argument(
        "--clip",
        type=float,
        nargs=2,
        default=None,
        help="Intensity clip range, --clip -1000 600 for CT HU."
    )
    parser.add_argument(
        "--zscore",
        action="store_true",
        help="Apply z-score normalization after clipping."
    )
    parser.add_argument(
        "--crop_mode",
        type=str,
        default="label",
        choices=["none", "label", "body"],
        help="Cropping mode: 'label' (crop to label bbox), 'body' (non-zero img), or 'none'."
    )
    parser.add_argument(
        "--crop_margin",
        type=int,
        default=10,
        help="Margin (in voxels) to add around the crop bounding box."
    )

    args = parser.parse_args()

    images_dir = Path(args.images)
    labels_dir = Path(args.labels) if args.labels is not None else None
    out_images_dir = Path(args.out_images)
    out_labels_dir = Path(args.out_labels) if args.out_labels is not None else None

    if labels_dir is None and args.crop_mode == "label":
        raise ValueError("crop_mode='label' requires --labels directory.")

    img_files = sorted(
        [
            p for p in images_dir.iterdir()
            if p.is_file() and (p.name.endswith(".nii") or p.name.endswith(".nii.gz"))
        ]
    )

    if not img_files:
        raise RuntimeError(f"No NIfTI files found in {images_dir}")

    for img_path in img_files:
        # Strip extension to get a clean base name image_001
        img_name = img_path.name
        base = img_name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]

        lbl_path = None
        if labels_dir is not None:
            candidates = []

            # 1) Same base name in labels dir
            candidates.append(labels_dir / (base + ".nii.gz"))
            candidates.append(labels_dir / (base + ".nii"))

            # 2) image_001 to label_001 pattern
            if base.startswith("image_"):
                idx = base[len("image_"):]  # "001"
                candidates.append(labels_dir / f"label_{idx}.nii.gz")
                candidates.append(labels_dir / f"label_{idx}.nii")

            # Pick the first existing candidate
            for cand in candidates:
                if cand.exists():
                    lbl_path = cand
                    break

            if lbl_path is None:
                raise FileNotFoundError(
                    f"Missing label for {img_path.name}. "
                    f"Tried: {[str(c) for c in candidates]}"
                )

        preprocess_pair(
            img_path=img_path,
            lbl_path=lbl_path,
            out_img_dir=out_images_dir,
            out_lbl_dir=out_labels_dir,
            target_spacing=args.target_spacing,
            clip_range=args.clip,
            # Don't z-score if doing percentile_norm
            do_zscore=args.zscore and not args.percentile_norm,
            crop_mode=args.crop_mode,
            crop_margin=args.crop_margin,
            percentile_norm=args.percentile_norm,
            percentile_range=tuple(args.percentile_range),
        )



if __name__ == "__main__":
    main()

In [ ]:
""" Script to perform inference with vesselFM."""

import logging
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import hydra
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

import numpy as np
import json
import nibabel as nib
from .cldice_utils import hard_cldice
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from monai.inferers import SlidingWindowInfererAdapt
from skimage.morphology import remove_small_objects
from skimage.exposure import equalize_hist

from vesselfm.seg.utils.data import generate_transforms
from vesselfm.seg.utils.io import determine_reader_writer
from vesselfm.seg.utils.evaluation import Evaluator, calculate_mean_metrics

from omegaconf import OmegaConf
from pathlib import Path


warnings.filterwarnings("ignore")
logger = logging.getLogger(__name__)

def build_model(num_classes=3, dropout=0.0):
    # Load inference config to get the same model definition with ckpt_path
    here = Path(__file__).resolve().parent
    config_dir = here / "configs"

    # Compose the full inference config
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=str(config_dir), job_name="av_model"):
        cfg_inf = compose(config_name="inference")

    if "model" not in cfg_inf:
        raise ValueError(
            f"'model' key not found in composed config. Top-level keys: {list(cfg_inf.keys())}"
        )

    mcfg = cfg_inf.model

    # Set number of output channels to num_classes
    if "out_channels" in mcfg:
        logger.info(f"[build_model] Setting model.out_channels -> {num_classes}")
        mcfg.out_channels = num_classes
    elif "num_classes" in mcfg:
        logger.info(f"[build_model] Setting model.num_classes -> {num_classes}")
        mcfg.num_classes = num_classes
    else:
        logger.warning(
            "[build_model] Neither 'out_channels' nor 'num_classes' found in model config; "
            "leaving output channels as-is."
        )

    # Dropout override
    if "dropout" in mcfg:
        logger.info(f"[build_model] Setting model.dropout -> {dropout}")
        mcfg.dropout = dropout

    # Instantiate MONAI DynUNet
    model = hydra.utils.instantiate(cfg_inf.model)

    # Load pretrained VesselFM weights
    try:
        logger.info(f"[build_model] Loading pretrained weights from {cfg_inf.ckpt_path}.")
        ckpt = torch.load(Path(cfg_inf.ckpt_path), map_location="cpu", weights_only=True)
    except Exception as e:
        logger.info(
            f"[build_model] Could not load ckpt from cfg_inf.ckpt_path ({e}). "
            "Falling back to Hugging Face vesselFM_base.pt."
        )
        hf_hub_download(repo_id="bwittmann/vesselFM", filename="meta.yaml")
        ckpt = torch.load(
            hf_hub_download(repo_id="bwittmann/vesselFM", filename="vesselFM_base.pt"),
            map_location="cpu",
            weights_only=True,
        )

    # Drop old head weights (1-channel) to avoid size mismatch
    head_keys = [k for k in ckpt.keys() if k.startswith("output_block.")]
    if head_keys:
        logger.info(
            f"[build_model] Removing {len(head_keys)} head params from checkpoint "
            f"to accommodate new 3-class head: {head_keys}"
        )
        for k in head_keys:
            ckpt.pop(k)

    # Now load backbone weights (strict=False allows missing head params)
    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    logger.info(
        f"[build_model] Loaded pretrained VesselFM weights with "
        f"{len(missing)} missing and {len(unexpected)} unexpected keys "
        f"(expected when swapping to a 3-class head)."
    )

    # Add an explicit vessel head as a second physical head.
    if not hasattr(model, "vessel_head"):
        logger.info("[build_model] Adding 1x1x1 vessel_head on top of A/V logits.")
        model.vessel_head = nn.Conv3d(num_classes, 1, kernel_size=1)

    return model


def load_model(cfg, device):
    try:
        logger.info(f"Loading model from {cfg.ckpt_path}.")
        ckpt = torch.load(Path(cfg.ckpt_path), map_location=device, weights_only=True)
    except:
        logger.info(f"Loading model from Hugging Face.")
        hf_hub_download(repo_id='bwittmann/vesselFM', filename='meta.yaml') # required to track downloads
        ckpt = torch.load(
            hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt'),
            map_location=device, weights_only=True
        )

    model = hydra.utils.instantiate(cfg.model)
    model.load_state_dict(ckpt, strict=False)
    return model

def get_paths(cfg):
    """
    Collect image and mask paths.

    Supports config layouts:
      - cfg.image_dir / cfg.mask_dir
      - cfg.image_path / cfg.mask_path
      - cfg.data.image_dir / cfg.data.mask_dir
      - cfg.data.image_path / cfg.data.mask_path

    Supports filename conventions:
      1) Same-name masks:
           image_004.nii.gz -> image_004.nii.gz
      2) image/label naming:
           image_004.nii.gz -> label_004.nii.gz
    """
    import os

    # --- 1. Read directories from config with fallbacks ---
    image_dir_str = (
        OmegaConf.select(cfg, "data.image_dir")
        or OmegaConf.select(cfg, "image_dir")
        or OmegaConf.select(cfg, "data.image_path")
        or OmegaConf.select(cfg, "image_path")
    )

    mask_dir_str = (
        OmegaConf.select(cfg, "data.mask_dir")
        or OmegaConf.select(cfg, "mask_dir")
        or OmegaConf.select(cfg, "data.mask_path")
        or OmegaConf.select(cfg, "mask_path")
    )

    if image_dir_str is None:
        raise RuntimeError(
            "image directory not set in config "
            "(looked for 'image_dir', 'data.image_dir', "
            "'image_path', and 'data.image_path')."
        )

    image_dir = Path(image_dir_str)

    # Normalize mask_dir: either a Path or None
    if mask_dir_str is None or mask_dir_str in ("", "null"):
        mask_dir = None
    else:
        mask_dir = Path(mask_dir_str)

    # --- 2. Collect images as Path objects ---
    # Use *.nii* so it works for .nii and .nii.gz
    image_paths = sorted(image_dir.glob("*.nii*"))
    if not image_paths:
        raise RuntimeError(f"No images found in {image_dir}")

    # If no mask_dir (pure inference), just return images
    if mask_dir is None:
        return image_paths, None

    # --- 3. Build mask paths with both naming schemes (also as Path objects) ---
    mask_paths = []

    for img_path in image_paths:
        # img_path is a Path
        img_name = img_path.name  # e.g. "image_004.nii.gz"

        # (a) First try: mask has EXACT same basename as image
        same_name_mask = mask_dir / img_name
        if same_name_mask.exists():
            mask_paths.append(same_name_mask)
            continue

        # (b) Second try: image_XXX.nii.gz -> label_XXX.nii.gz
        alt_mask = None
        if img_name.startswith("image_"):
            suffix = img_name[len("image_"):]          # "004.nii.gz"
            alt_mask = mask_dir / f"label_{suffix}"
            if alt_mask.exists():
                mask_paths.append(alt_mask)
                continue

        # If we get here, no matching mask was found for this image
        msg = (
            f"Could not find a mask for image:\n  {img_path}\n"
            f"Tried:\n  {same_name_mask}"
        )
        if alt_mask is not None:
            msg += f"\n  {alt_mask}"
        raise FileNotFoundError(msg)

    return image_paths, mask_paths



def resample(image, factor=None, target_shape=None):
    if factor == 1:
        return image
    
    if target_shape:
        _, _, new_d, new_h, new_w = target_shape
    else:
        _, _, d, h, w = image.shape
        new_d, new_h, new_w = int(round(d / factor)), int(round(h / factor)), int(round(w / factor))
    return F.interpolate(image, size=(new_d, new_h, new_w), mode="trilinear", align_corners=False)

@hydra.main(config_path="configs", config_name="inference", version_base="1.3.2")
def main(cfg):
    # seed libraries
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    torch.cuda.manual_seed_all(cfg.seed)

    # set device
    logger.info(f"Using device {cfg.device}.")
    device = cfg.device

    # load model and ckpt
    model = load_model(cfg, device)
    model.to(device)
    model.eval()

    # init pre-processing transforms
    transforms = generate_transforms(cfg.transforms_config)

    # i/o
    output_folder = Path(cfg.output_folder)
    output_folder.mkdir(exist_ok=True)

    image_paths, mask_paths = get_paths(cfg)
    logger.info(f"Found {len(image_paths)} images in {cfg.image_path}.")

    file_ending = (cfg.image_file_ending if cfg.image_file_ending else image_paths[0].suffix)
    image_reader_writer = determine_reader_writer(file_ending)()
    save_writer = determine_reader_writer(file_ending)()

    # init sliding window inferer
    logger.debug(f"Sliding window patch size: {cfg.patch_size}")
    logger.debug(f"Sliding window batch size: {cfg.batch_size}.")
    logger.debug(f"Sliding window overlap: {cfg.overlap}.")
    inferer = SlidingWindowInfererAdapt(
        roi_size=cfg.patch_size, sw_batch_size=cfg.batch_size, overlap=cfg.overlap, 
        mode=cfg.mode, sigma_scale=cfg.sigma_scale, padding_mode=cfg.padding_mode
    )

    # loop over images
    metrics_dict = {}
    with torch.no_grad():
        for idx, image_path in tqdm(
            enumerate(image_paths),
            total=len(image_paths),
            desc="Processing images.",
        ):
            preds = []  # per-scale logits
            mask_np = None

            for scale in cfg.tta.scales:
                # read image (and mask if available)
                image_np = image_reader_writer.read_images(image_path)[0].astype(np.float32)
                image = transforms(image_np)[None].to(device)

                if mask_paths is not None and mask_np is None:
                    # Load 3-class GT: 0=bg,1=artery,2=vein
                    mask_np = image_reader_writer.read_images(mask_paths[idx])[0].astype(np.int16)

                # TTA intensity transforms
                if cfg.tta.invert:
                    if image.mean() > cfg.tta.invert_mean_thresh:
                        image = 1 - image
                if cfg.tta.equalize_hist:
                    image_np = image.cpu().squeeze().numpy()
                    image_equal_hist_np = equalize_hist(image_np, nbins=cfg.tta.hist_bins)
                    image = torch.from_numpy(image_equal_hist_np).to(image.device)[None][None]

                # resample for scale, run model, resample back
                original_shape = image.shape
                image_scaled = resample(image, factor=scale)
                logits = inferer(image_scaled, model)                     # (1,3,D,H,W)
                logits = resample(logits, target_shape=original_shape)    # back to original patch grid
                preds.append(logits.cpu().squeeze())                      # (3,D,H,W)

            # Preds is a list of per-scale logits, each (3,D,H,W)
            logits_ensemble = torch.stack(preds).mean(dim=0)    # (3,D,H,W)

            if hasattr(model, "av_refine_head"):
                # Same combination as in eval_epoch (but single volume)
                base_probs = F.softmax(logits_ensemble.unsqueeze(0), dim=1)  # (1,3,D,H,W)
                p_bg = base_probs[:, 0:1, ...]
                p_union = base_probs[:, 1:3, ...].sum(dim=1, keepdim=True).clamp(0.0, 1.0)

                av_logits = model.av_refine_head(logits_ensemble.unsqueeze(0))  # (1,2,D,H,W)
                av_probs = F.softmax(av_logits, dim=1)
                p_art_cond = av_probs[:, 0:1, ...]
                p_vein_cond = av_probs[:, 1:2, ...]

                p_art = p_union * p_art_cond
                p_vein = p_union * p_vein_cond

                denom = p_bg + p_art + p_vein + 1e-8
                probs_final = torch.cat(
                    [p_bg / denom, p_art / denom, p_vein / denom],
                    dim=1,
                )[0]                                  # (3,D,H,W)
            else:
                # Fallback: original behavior
                if cfg.merging.max:
                    probs_final = torch.stack([F.softmax(p, dim=0) for p in preds]).max(dim=0)[0]
                else:
                    probs_final = torch.stack([F.softmax(p, dim=0) for p in preds]).mean(dim=0)

            label = probs_final.argmax(0).cpu().numpy().astype(np.uint8)

            # Class-wise CC cleanup
            if cfg.post.apply:
                cleaned = np.zeros_like(label, dtype=np.uint8)
                for c in (1, 2):  # artery, vein
                    cm = (label == c)
                    cm = remove_small_objects(
                        cm,
                        min_size=cfg.post.small_objects_min_size,
                        connectivity=cfg.post.small_objects_connectivity,
                    )
                    cleaned[cm] = c
                label = cleaned

            # Label is a numpy array (D, H, W) in model/reader order
            label_np = label.astype(np.uint8)

            # Load the original CT as reference for affine + header
            ref_nii = nib.load(str(image_path))
            ref_shape = ref_nii.shape

            # If shape is (D,H,W)=(234,186,247) but ref is (247,186,234),
            # swap axes 0 and 2 so we match nibabel's (X,Y,Z).
            if (
                label_np.shape != ref_shape and
                label_np.shape[0] == ref_shape[2] and
                label_np.shape[1] == ref_shape[1] and
                label_np.shape[2] == ref_shape[0]
            ):
                # (D,H,W) -> (W,H,D) == (X,Y,Z)
                label_np = np.transpose(label_np, (2, 1, 0))
                logger.info(
                    f"Transposed prediction from original shape to match ref shape {ref_shape}."
                )

            pred_nii = nib.Nifti1Image(
                label_np,
                affine=ref_nii.affine,
                header=ref_nii.header,
            )

            # Keep sform/qform consistent
            pred_nii.set_sform(
                ref_nii.get_sform(),
                code=ref_nii.get_sform(coded=True)[1] or 1
            )
            pred_nii.set_qform(
                ref_nii.get_qform(),
                code=ref_nii.get_qform(coded=True)[1] or 1
            )

            # Make sure sform/qform are consistent
            pred_nii.set_sform(ref_nii.get_sform(), code=ref_nii.get_sform(coded=True)[1] if ref_nii.get_sform(coded=True)[1] else 1)
            pred_nii.set_qform(ref_nii.get_qform(), code=ref_nii.get_qform(coded=True)[1] if ref_nii.get_qform(coded=True)[1] else 1)

            out_path = output_folder / f"{image_path.name.split('.')[0]}_{cfg.file_app}pred.nii.gz"
            nib.save(pred_nii, str(out_path))

            # Metrics if GT masks are available
            if mask_paths is not None and mask_np is not None:
                """
                label: (D, H, W) with {0:bg, 1:artery, 2:vein}
                mask_np: (D, H, W) with {0:bg, 1:artery, 2:vein}
                """

                # UNION (A ∪ V)
                union_pred = label > 0               # (D,H,W) bool
                union_gt   = mask_np > 0             # (D,H,W) bool

                inter_u = np.logical_and(union_pred, union_gt).sum()
                denom_u = union_pred.sum() + union_gt.sum()
                dice_union = 2.0 * inter_u / (denom_u + 1e-5) if denom_u > 0 else 0.0
                cldice_union = hard_cldice(union_pred.astype(bool), union_gt.astype(bool))

                # ARTERY (class = 1)
                g_art = (mask_np == 1)
                if g_art.any():
                    p_art = (label == 1)

                    inter_a = np.logical_and(p_art, g_art).sum()
                    denom_a = p_art.sum() + g_art.sum()
                    dice_art = 2.0 * inter_a / (denom_a + 1e-5) if denom_a > 0 else 0.0
                    cldice_art = hard_cldice(p_art.astype(bool), g_art.astype(bool))
                else:
                    dice_art = 0.0
                    cldice_art = 0.0

                # VEIN (class = 2)
                g_vein = (mask_np == 2)
                if g_vein.any():
                    p_vein = (label == 2)

                    inter_v = np.logical_and(p_vein, g_vein).sum()
                    denom_v = p_vein.sum() + g_vein.sum()
                    dice_vein = 2.0 * inter_v / (denom_v + 1e-5) if denom_v > 0 else 0.0
                    cldice_vein = hard_cldice(p_vein.astype(bool), g_vein.astype(bool))
                else:
                    dice_vein = 0.0
                    cldice_vein = 0.0

                case_name = image_path.name.split(".")[0]
                logger.info(
                    f"{case_name}: "
                    f"Dice(A∪V)={dice_union:.4f} clDice(A∪V)={cldice_union:.4f} "
                    f"Dice(art)={dice_art:.4f} clDice(art)={cldice_art:.4f} "
                    f"Dice(vein)={dice_vein:.4f} clDice(vein)={cldice_vein:.4f}"
                )

                # Store all six metrics
                metrics_dict[case_name] = {
                    "dice":          torch.tensor(dice_union),
                    "cldice":        torch.tensor(cldice_union),
                    "dice_art":      torch.tensor(dice_art),
                    "cldice_art":    torch.tensor(cldice_art),
                    "dice_vein":     torch.tensor(dice_vein),
                    "cldice_vein":   torch.tensor(cldice_vein),
                }

    # Summarize over all images
    if mask_paths is not None and len(metrics_dict) > 0:
        # Compute mean for every metric key we stored
        metric_names = list(next(iter(metrics_dict.values())).keys())
        mean_metrics = {}
        for m in metric_names:
            vals = [metrics_dict[k][m].item() for k in metrics_dict]
            mean_metrics[m] = float(np.mean(vals))

        logger.info(f"Mean Dice(A∪V): {mean_metrics['dice']:.4f}")
        logger.info(f"Mean clDice(A∪V): {mean_metrics['cldice']:.4f}")
        logger.info(f"Mean Dice(art): {mean_metrics['dice_art']:.4f}")
        logger.info(f"Mean clDice(art): {mean_metrics['cldice_art']:.4f}")
        logger.info(f"Mean Dice(vein): {mean_metrics['dice_vein']:.4f}")
        logger.info(f"Mean clDice(vein): {mean_metrics['cldice_vein']:.4f}")

        with open(output_folder / "metrics_per_volume.json", "w") as f:
            json.dump(
                {k: {m: float(v[m].item()) for m, v_m in v.items()} for k, v in metrics_dict.items()},
                f,
                indent=2,
            )

        with open(output_folder / "metrics_mean.json", "w") as f:
            json.dump(mean_metrics, f, indent=2)

if __name__ == "__main__":
    main()